In [ ]:
# AI 모델 학습 데이터 필터링

**기능:**
- Google Drive에 저장된 이미지 데이터셋을 필터링합니다.
- 아래 3가지 조건에 맞지 않는 이미지를 걸러냅니다.
    1. 사진이 아닌 그림/일러스트
    2. 사람 사진이 아닌 이미지 (얼굴이 없는 경우)
    3. 2명 이상의 얼굴이 있는 사진

**프로세스:**
1. `face_recognition` 라이브러리로 이미지 내 얼굴 개수를 탐지합니다.
2. 얼굴이 1개가 아니면 `data_rejected` 폴더로 이동합니다.
3. OpenAI의 CLIP 모델을 사용하여 이미지가 사진인지 그림인지 분류합니다.
4. 그림으로 분류되면 `data_rejected` 폴더로, 사진이면 `data_filtered` 폴더로 이동합니다.


In [ ]:
# -*- coding: utf-8 -*-
"""
### 1. 패키지 설치

필터링 작업에 필요한 라이브러리를 설치합니다.
- face_recognition: dlib 기반의 얼굴 인식 라이브러리
- transformers, torch: OpenAI CLIP 모델 사용을 위함
- Pillow: 이미지 처리
"""

!pip install face_recognition transformers torch torchvision Pillow tqdm


In [ ]:
# -*- coding: utf-8 -*-
"""
### 2. Google Drive 마운트 및 GPU 확인
"""
from google.colab import drive
import torch

try:
    drive.mount('/content/drive')
    print("✅ Google Drive 연결 완료")
except Exception as e:
    print(f"❌ Google Drive 연결 실패: {e}")

# GPU 확인
if torch.cuda.is_available():
    print("🚀 GPU 가속 사용 가능")
    device = "cuda"
else:
    print("⚠️ GPU 사용 불가 - CPU 모드로 실행")
    device = "cpu"


In [ ]:
# -*- coding: utf-8 -*-
"""
### 3. 라이브러리 임포트 및 경로 설정
"""
import os
import shutil
from PIL import Image
import face_recognition
import torch
from transformers import CLIPProcessor, CLIPModel
from tqdm.auto import tqdm

# 경로 설정 (사용자 요청에 따라 수정)
BASE_DIR = "/content/drive/MyDrive/WhosYourAncestor_Selfie_Images"
SOURCE_DATA_DIR = BASE_DIR
FILTERED_DATA_DIR = "/content/drive/MyDrive/WhosYourAncestor_Selfie_Images_filtered"
REJECTED_DATA_DIR = "/content/drive/MyDrive/WhosYourAncestor_Selfie_Images_rejected"

# 12개 국가 리스트
COUNTRIES = [
    "british", "chinese", "ethiopian", "french", "indian",
    "indigenous", "japanese", "korean", "mexican",
    "nigerian", "russian", "saudi"
]

# 결과 폴더 생성
os.makedirs(FILTERED_DATA_DIR, exist_ok=True)
os.makedirs(REJECTED_DATA_DIR, exist_ok=True)

print("✅ 경로 설정 완료")
print("SOURCE_DATA_DIR:", SOURCE_DATA_DIR)
print("FILTERED_DATA_DIR:", FILTERED_DATA_DIR)
print("REJECTED_DATA_DIR:", REJECTED_DATA_DIR)


In [ ]:
# -*- coding: utf-8 -*-
"""
### 4. CLIP 모델 로드

이미지가 사진인지 그림인지 분류하기 위해 OpenAI의 CLIP 모델을 로드합니다.
"""
model_name = "openai/clip-vit-base-patch32"
try:
    clip_processor = CLIPProcessor.from_pretrained(model_name)
    clip_model = CLIPModel.from_pretrained(model_name).to(device)
    print("✅ CLIP 모델 로드 완료")
except Exception as e:
    print(f"❌ CLIP 모델 로드 실패: {e}")


In [ ]:
# -*- coding: utf-8 -*-
"""
### 5. 이미지 필터링 함수 정의

실제 필터링을 수행하는 함수입니다.
"""

def is_drawing(image_path, threshold=0.5):
    """
    CLIP 모델을 사용하여 이미지가 사진인지 그림인지 판단합니다.
    (로직 개선: 프롬프트 일반화)
    """
    try:
        image = Image.open(image_path).convert("RGB")
        
        # 텍스트 프롬프트 (더 일반적인 표현으로 수정)
        prompts = ["a photograph of a person", "a drawing, illustration, or anime of a person"]
        
        inputs = clip_processor(text=prompts, images=image, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)
        
        is_drawing_prob = probs[0][1].item()
        
        return is_drawing_prob > threshold, is_drawing_prob

    except Exception as e:
        print(f"\\n⚠️ CLIP 모델 추론 실패: {os.path.basename(image_path)} - {e}")
        return True, 1.0

def filter_images_for_country(country):
    """
    한 국가의 모든 이미지에 대해 필터링을 수행합니다.
    (로직 개선: face_recognition 모델 변경 및 상세 로그 추가)
    """
    print(f"\\n--- {country.upper()} 국가 필터링 시작 ---")
    
    source_country_dir = os.path.join(SOURCE_DATA_DIR, country)
    filtered_country_dir = os.path.join(FILTERED_DATA_DIR, country)
    rejected_country_dir = os.path.join(REJECTED_DATA_DIR, country)

    os.makedirs(filtered_country_dir, exist_ok=True)
    os.makedirs(rejected_country_dir, exist_ok=True)
    
    if not os.path.exists(source_country_dir):
        print(f"⚠️ 소스 폴더가 없습니다: {source_country_dir}")
        return

    image_files = [f for f in os.listdir(source_country_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    
    if not image_files:
        print("🤷‍♀️ 처리할 이미지가 없습니다.")
        return

    print(f"총 {len(image_files)}개의 이미지 처리 시작...")
    total_filtered = 0
    total_rejected = 0

    for filename in tqdm(image_files, desc=f"Filtering {country}"):
        source_path = os.path.join(source_country_dir, filename)
        if not os.path.exists(source_path):
             continue
        
        rejected = False
        rejection_reason = ""

        try:
            # 1. 얼굴 개수 확인 (hog 모델로 변경)
            image_array = face_recognition.load_image_file(source_path)
            face_locations = face_recognition.face_locations(image_array, model='hog') 
            
            if len(face_locations) != 1:
                rejected = True
                rejection_reason = f"얼굴 개수 {len(face_locations)}개"
            else:
                # 2. 사진/그림 구분
                is_art, probability = is_drawing(source_path)
                if is_art:
                    rejected = True
                    rejection_reason = f"그림일 확률 {probability:.2f}"
            
            if rejected:
                shutil.move(source_path, os.path.join(rejected_country_dir, filename))
                total_rejected += 1
                tqdm.write(f"❌ 제외: {filename} ({rejection_reason})")
            else:
                # 모든 필터 통과
                shutil.move(source_path, os.path.join(filtered_country_dir, filename))
                total_filtered += 1
                tqdm.write(f"✅ 통과: {filename}")

        except Exception as e:
            try:
                shutil.move(source_path, os.path.join(rejected_country_dir, filename))
                total_rejected += 1
                tqdm.write(f"❌ 제외: {filename} (오류 발생: {e})")
            except Exception as move_e:
                print(f"\\n❌ 이동 실패: {filename} - {move_e}")
    
    print(f"\\n--- {country.upper()} 국가 필터링 완료 ---")
    print(f"✅ 통과: {total_filtered}개")
    print(f"❌ 제외: {total_rejected}개")


In [ ]:
# -*- coding: utf-8 -*-
"""
### 6. 필터링 실행

모든 국가에 대해 필터링을 실행합니다.
"""
for country in COUNTRIES:
    filter_images_for_country(country)

print("\\n\\n🎉 모든 국가에 대한 필터링 작업이 완료되었습니다.")
